# Part 5: More replicas, and where each request goes

Fill both routers first (`pytest tests/test_routing.py`). The runs, from the repo root, one at a time:

```bash
# how many replicas: Sarathi on 2 replicas, round-robin (about 5 minutes)
REPLICAS=2 GLOBAL=round_robin SCHED=sarathi CHUNK=512 QPS=5.53 bash scripts/run_default.sh traces/azure_conversation.csv

# routers on the long-short bursts trace (seconds each)
for G in round_robin lor lot; do
  REPLICAS=2 GLOBAL=$G SCHED=sarathi CHUNK=512 QPS=8 bash scripts/run_default.sh traces/long_short_bursts.csv
done

# LOT on the real trace (about 5 minutes)
REPLICAS=2 GLOBAL=lot SCHED=sarathi CHUNK=512 QPS=5.53 bash scripts/run_default.sh traces/azure_conversation.csv
```

In [2]:
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "traces").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from analysis import workload as wl
from analysis.replay import busy_percent, describe_run, load_run
from analysis.workload import load_trace, predict_queue_wait

azure = load_trace(ROOT / "traces" / "azure_conversation.csv")
bursts = load_trace(ROOT / "traces" / "long_short_bursts.csv")

def summary(run_dir, trace):
    run = load_run(run_dir, trace)
    tbt = run["tbt"].set_index("percentile").seconds
    req = run["requests"]
    return {"wait_p99": req.wait.quantile(0.99), "e2e_p99": req.e2e.quantile(0.99),
            "tbt_p99_ms": tbt[99] * 1e3, "busy": busy_percent(run_dir)}

## How many replicas?

In [3]:
for n in (1, 2, 3):
    w = predict_queue_wait(azure, prefill_tok_per_s=n * wl.PREFILL_TOK_PER_S,
                           decode_tok_per_s=n * wl.DECODE_TOK_PER_S)
    print(f"{n} replica(s): predicted P99 wait {np.quantile(w, 0.99):.2f} s")

ONE_REPLICA = ROOT / "simulator_output" / "2026-10-27_11-25-39-522190"   # from Part 3
TWO_RR = ROOT / "simulator_output" / "2026-10-27_11-17-19-030452"
for d in (ONE_REPLICA, TWO_RR):
    s = summary(d, azure)
    print(f"{describe_run(d):42s} P99 wait {s['wait_p99']:6.2f} s   P99 time between tokens {s['tbt_p99_ms']:.0f} ms")

1 replica(s): predicted P99 wait 72.13 s
2 replica(s): predicted P99 wait 0.38 s
3 replica(s): predicted P99 wait 0.15 s
sarathi C=512                              P99 wait 171.22 s   P99 time between tokens 62 ms
sarathi C=512, 2 replicas, round_robin     P99 wait   0.60 s   P99 time between tokens 54 ms


## Routers on the long-short bursts trace

In [4]:
BURST_RUNS = [ROOT / "simulator_output" / d for d in [
    "2026-10-27_11-34-59-967536", "2026-10-27_11-35-09-810665", "2026-10-27_11-35-18-785159",
]]
for d in BURST_RUNS:
    s = summary(d, bursts)
    print(f"{describe_run(d):42s} wait P99 {s['wait_p99']:5.2f} s   end-to-end P99 {s['e2e_p99']:5.2f} s   "
          f"busy % (replica 1 / 2) {' / '.join(f'{b:.0f}' for b in s['busy'])}")

sarathi C=512, 2 replicas, round_robin     wait P99  8.99 s   end-to-end P99  9.78 s   busy % (replica 1 / 2) 100 / 5
sarathi C=512, 2 replicas, lor             wait P99  1.14 s   end-to-end P99  1.98 s   busy % (replica 1 / 2) 80 / 76
sarathi C=512, 2 replicas, lot             wait P99  0.77 s   end-to-end P99  0.85 s   busy % (replica 1 / 2) 81 / 81


## Round-robin vs LOT on the real trace

In [6]:
TWO_LOT = ROOT / "simulator_output" / "2026-10-27_11-38-59-639477"
for d in (TWO_RR, TWO_LOT):
    s = summary(d, azure)
    print(f"{describe_run(d):42s} P99 wait {s['wait_p99']:5.2f} s   P99 time between tokens {s['tbt_p99_ms']:.0f} ms")

sarathi C=512, 2 replicas, round_robin     P99 wait  0.60 s   P99 time between tokens 54 ms
sarathi C=512, 2 replicas, lot             P99 wait  0.40 s   P99 time between tokens 54 ms


## Questions

**Q1.** Fill in the table. What is the smallest number of replicas that meets both
targets (P99 wait under 2 s, P99 time between tokens under 100 ms)? How
close was your Part 1 model to Vidur?

| Replicas | Predicted P99 wait (Part 1 model) | Vidur P99 wait | Vidur P99 time between tokens |
|---|---|---|---|
| 1 | 72.13 s|171.22 s (Part 3, C = 512) | 62 ms|
| 2 | 0.38 s|0.60 s  |54 ms |
| 3 | 0.15 s| — | — |

**Your answer:**
The smallest nuber of replicas that meets both targets is 2. Besides the first replicas, the part 1 prediction model was quite similar to the Vidur model. However, in both cases, the part 1 model was more optimistic and as a result, undershoot the time.

**Q2.** Fill in the table for the long-short bursts trace. Which router balances the
two replicas best, and which worst? Explain each router's busy split and wait. The worked example shows one burst on empty replicas: what changes when later bursts arrive while earlier requests are still outstanding? (Replicas 1 / 2 in the output are replica ids 0 / 1.)

| Router | Wait P99 | End-to-end P99 | Busy % (replica 1 / 2) |
|---|---|---|---|
| Round-robin |8.99 s |9.78 s |	100 / 5 |
| LOR |1.14 s |	1.98 s |	80 / 76 |
| LOT |0.77 s |	0.85 s |	81 / 81 |

**Your answer:**
The LOT router balance the two replicas the best and Round-robin was the worse rotuer.

In round-robin, the router ignores request size. As a result, when recieving a long-short prompt pattern, one replica will recieve all the long request and one replica will reciee all the short request causing a mssaive imbalacne on how busy each router is.

In LOR, the router picks the replica with the fewest oustanding request. As a result, it's not nearly imbalance as round-robin which keeps each router busy. However, this router still ignores request size, so it counts long request the same as short requst. As a result, some imbalance, though not as high as round-roblin, still occured between the two replicas.

In LOT, the router picks the replica with the fewest outstanding tokens. This finer grain provides the best balance between the two replicas resulting in equal busy work.

On empty replicas, every replicas "look" the same since LOR and LOT have no outstanding tokens or request to respond to.

**Q3.** Vidur's original LOR counted only requests **waiting** in each replica's
queue, not running ones. On the real trace with 2 replicas, those queues are
almost always empty. Which replica would that router pick for almost every
request, and why? Which definition is better, and why?

**Your answer:**
The router would pick replica 0. Since both router are almost always empty, both replicas would "look" the same. Therefore, in a tiebreaker, the router would pick the replica with lower id (0). A better definition would be counting both the outstanding and running request because this provides a greater accuracy and representation on how busy each router is.

**Q4.** On the real trace with 2 replicas, compare round-robin with LOT: P99 wait
and P99 time between tokens. Does the router matter here? In one sentence:
when is a load-aware router worth it?

**Your answer:**
The router somewhat matter. There is a 0.2s P99 wait difference between round-robin and LOT and both routers have identical P99 time. A load-aware router is worth when the requests are uneven pattern but less worth it for requests that are similar size.